# Week 80, Day 3: Project: endpoints and auth

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w080d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')
with open('helpdesk_domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s rules, with no web or database code (Week 80, Day 1).\n\nEverything here is plain data and pure functions, so it can be tested on its own and used by the\ndatabase layer, the web layer and the tests alike.\n"""\nimport re\nfrom http import HTTPStatus\n\nROLES = ("customer", "agent", "admin")\nPRIORITIES = ("low", "normal", "high", "urgent")\nSTATUSES = ("open", "in_progress", "waiting", "resolved", "closed")\nSTAFF = {"agent", "admin"}\n\n# status -> {new status: roles that may make the change}\nTRANSITIONS = {\n    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},\n    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},\n    "waiting": {"in_progress": {"customer", "agent", "admin"}},\n    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},\n    "closed": {},\n}\n\n# action -> (roles always allowed, roles allowed only on their own tickets)\nPERMISSIONS = {\n    "create_ticket": ({"customer"}, set()),\n    "view_ticket": ({"agent", "admin"}, {"customer"}),\n    "list_all_tickets": ({"agent", "admin"}, set()),\n    "edit_ticket": ({"admin"}, {"customer"}),\n    "comment": ({"agent", "admin"}, {"customer"}),\n    "internal_note": ({"agent", "admin"}, set()),\n    "change_priority": ({"agent", "admin"}, set()),\n    "assign": ({"agent", "admin"}, set()),\n    "delete_ticket": ({"admin"}, set()),\n    "view_stats": ({"agent", "admin"}, set()),\n}\n\n\nclass ValidationError(Exception):\n    """Raised with a dict of field -> message."""\n\n    def __init__(self, errors):\n        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))\n        self.errors = errors\n\n\ndef validate_ticket(data, partial=False):\n    """Check a ticket body from a client. partial=True for PATCH: only the fields present are checked."""\n    if not isinstance(data, dict):\n        raise ValidationError({"_": "expected a JSON object"})\n    errors, clean = {}, {}\n    unknown = sorted(set(data) - {"title", "body", "priority"})\n    if unknown:\n        errors["_"] = f"unexpected field(s): {\', \'.join(unknown)}"\n    for field in ("title", "body"):\n        if field not in data:\n            if not partial:\n                errors[field] = "required"\n            continue\n        value = data[field]\n        limit = (3, 120) if field == "title" else (1, 5000)\n        if not isinstance(value, str) or not limit[0] <= len(value.strip()) <= limit[1]:\n            errors[field] = f"{limit[0]} to {limit[1]} characters"\n        else:\n            clean[field] = value.strip()\n    if "priority" in data:\n        if data["priority"] not in PRIORITIES:\n            errors["priority"] = f"one of {\', \'.join(PRIORITIES)}"\n        else:\n            clean["priority"] = data["priority"]\n    elif not partial:\n        clean["priority"] = "normal"\n    if partial and not data:\n        errors["_"] = "nothing to change"\n    if errors:\n        raise ValidationError(errors)\n    return clean\n\n\ndef can_transition(current, new, role):\n    """True if someone with this role may move a ticket from `current` to `new`."""\n    return role in TRANSITIONS.get(current, {}).get(new, set())\n\n\ndef allowed(role, action, is_owner=False):\n    """True if the role may do the action (is_owner: the ticket is the user\'s own)."""\n    always, own = PERMISSIONS[action]\n    return role in always or (is_owner and role in own)\n\n\ndef problem(status, detail, title=None, **extra):\n    """An RFC 9457 problem-details response: (body, status, headers)."""\n    body = {"type": "about:blank", "title": title or HTTPStatus(status).phrase, "status": status, "detail": detail}\n    body.update(extra)\n    return body, status, {"Content-Type": "application/problem+json"}\n\n\nROUTES = [\n    ("POST", "/users", "sign up as a customer", 201),\n    ("POST", "/tokens", "log in and get a token", 201),\n    ("GET", "/me", "who am I", 200),\n    ("GET", "/tickets", "list tickets (customers see their own)", 200),\n    ("POST", "/tickets", "open a ticket", 201),\n    ("GET", "/tickets/{ticket_id}", "read a ticket", 200),\n    ("PATCH", "/tickets/{ticket_id}", "change a ticket\'s title, body or priority", 200),\n    ("DELETE", "/tickets/{ticket_id}", "delete a ticket (admins)", 204),\n    ("POST", "/tickets/{ticket_id}/transitions", "change a ticket\'s status", 200),\n    ("PUT", "/tickets/{ticket_id}/assignee", "assign a ticket to an agent", 200),\n    ("GET", "/tickets/{ticket_id}/comments", "list a ticket\'s comments", 200),\n    ("POST", "/tickets/{ticket_id}/comments", "add a comment or internal note", 201),\n    ("GET", "/stats", "ticket counts (staff)", 200),\n    ("GET", "/healthz", "is the process alive", 200),\n    ("GET", "/readyz", "can it serve requests", 200),\n]\n\n\ndef docs_table(routes):\n    """The routes as a Markdown table, for the README."""\n    lines = ["| Method | Path | What it does | Success |", "|---|---|---|---|"]\n    lines += [f"| {m} | `{p}` | {what} | {code} {HTTPStatus(code).phrase} |" for m, p, what, code in routes]\n    return "\\n".join(lines)\n')
with open('helpdesk_db.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s database: migrations and the queries the app needs (Week 80, Day 2).\n\nEvery query uses parameters; every change that touches more than one row runs in a transaction.\n"""\nimport sqlite3\n\nMIGRATIONS = [\n    (1, "users", """\n        CREATE TABLE users (\n            id INTEGER PRIMARY KEY,\n            email TEXT NOT NULL UNIQUE,\n            password TEXT NOT NULL,\n            role TEXT NOT NULL CHECK (role IN (\'customer\', \'agent\', \'admin\')),\n            created_at TEXT NOT NULL\n        )"""),\n    (2, "tickets", """\n        CREATE TABLE tickets (\n            id INTEGER PRIMARY KEY,\n            customer_id INTEGER NOT NULL REFERENCES users (id),\n            assignee_id INTEGER REFERENCES users (id),\n            title TEXT NOT NULL,\n            body TEXT NOT NULL,\n            priority TEXT NOT NULL DEFAULT \'normal\' CHECK (priority IN (\'low\', \'normal\', \'high\', \'urgent\')),\n            status TEXT NOT NULL DEFAULT \'open\'\n                CHECK (status IN (\'open\', \'in_progress\', \'waiting\', \'resolved\', \'closed\')),\n            created_at TEXT NOT NULL,\n            updated_at TEXT NOT NULL\n        )"""),\n    (3, "comments and events", """\n        CREATE TABLE comments (\n            id INTEGER PRIMARY KEY,\n            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,\n            author_id INTEGER NOT NULL REFERENCES users (id),\n            body TEXT NOT NULL,\n            internal INTEGER NOT NULL DEFAULT 0,\n            created_at TEXT NOT NULL\n        );\n        CREATE TABLE events (\n            id INTEGER PRIMARY KEY,\n            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,\n            actor_id INTEGER NOT NULL REFERENCES users (id),\n            kind TEXT NOT NULL,\n            detail TEXT NOT NULL,\n            at TEXT NOT NULL\n        )"""),\n    (4, "indexes", """\n        CREATE INDEX idx_tickets_status ON tickets (status, id);\n        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);\n        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),\n]\n\n\nclass ConflictError(Exception):\n    """The row changed since it was read."""\n\n\ndef connect(path=":memory:"):\n    conn = sqlite3.connect(path)\n    conn.row_factory = sqlite3.Row\n    conn.execute("PRAGMA foreign_keys = ON")\n    return conn\n\n\ndef migrate(conn, migrations=MIGRATIONS):\n    """Apply the migrations the database hasn\'t had yet, each in its own transaction. Returns their versions."""\n    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")\n    conn.commit()\n    versions = [v for v, _, _ in migrations]\n    if versions != list(range(1, len(versions) + 1)):\n        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")\n    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]\n    if current > len(versions):\n        raise RuntimeError(f"the database is at version {current}, newer than this code")\n    applied = []\n    for version, name, sql in migrations[current:]:\n        try:\n            conn.execute("BEGIN")\n            for statement in sql.split(";"):\n                if statement.strip():\n                    conn.execute(statement)\n            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))\n            conn.commit()\n        except Exception:\n            conn.rollback()\n            raise\n        applied.append(version)\n    return applied\n\n\ndef _dict(row):\n    return dict(row) if row is not None else None\n\n\ndef _event(conn, ticket_id, actor_id, kind, detail, now):\n    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",\n                 (ticket_id, actor_id, kind, detail, now))\n\n\ndef create_user(conn, email, password_hash, role, now):\n    try:\n        with conn:\n            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",\n                               (email, password_hash, role, now))\n    except sqlite3.IntegrityError as e:\n        if "UNIQUE" not in str(e):\n            raise\n        raise ValueError("email already registered") from None\n    return cur.lastrowid\n\n\ndef get_user(conn, user_id):\n    return _dict(conn.execute("SELECT id, email, role, created_at FROM users WHERE id = ?", (user_id,)).fetchone())\n\n\ndef get_user_by_email(conn, email):\n    return _dict(conn.execute("SELECT id, email, password, role FROM users WHERE email = ?", (email,)).fetchone())\n\n\ndef create_ticket(conn, customer_id, clean, now):\n    with conn:\n        cur = conn.execute(\n            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",\n            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))\n        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)\n    return get_ticket(conn, cur.lastrowid)\n\n\ndef get_ticket(conn, ticket_id):\n    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())\n\n\ndef _like(text):\n    return "%" + text.replace("\\\\", "\\\\\\\\").replace("%", "\\\\%").replace("_", "\\\\_") + "%"\n\n\ndef list_tickets(conn, customer_id=None, status=None, priority=None, assignee_id=None, q=None, after_id=0, limit=20):\n    """Tickets matching the filters, by id, after a cursor. Returns (items, next_after or None)."""\n    where, values = ["id > ?"], [after_id]\n    for column, value in (("customer_id", customer_id), ("status", status), ("priority", priority), ("assignee_id", assignee_id)):\n        if value is not None:\n            where.append(f"{column} = ?")\n            values.append(value)\n    if q:\n        where.append("title LIKE ? ESCAPE \'\\\\\'")\n        values.append(_like(q))\n    rows = conn.execute(f"SELECT * FROM tickets WHERE {\' AND \'.join(where)} ORDER BY id LIMIT ?", values + [limit + 1]).fetchall()\n    items = [dict(r) for r in rows[:limit]]\n    return items, (items[-1]["id"] if len(rows) > limit else None)\n\n\ndef update_ticket(conn, ticket_id, changes, actor_id, now):\n    if not changes:\n        return get_ticket(conn, ticket_id)\n    columns = sorted(changes)\n    with conn:\n        conn.execute(f"UPDATE tickets SET {\', \'.join(f\'{c} = ?\' for c in columns)}, updated_at = ? WHERE id = ?",\n                     [changes[c] for c in columns] + [now, ticket_id])\n        _event(conn, ticket_id, actor_id, "edited", ",".join(columns), now)\n    return get_ticket(conn, ticket_id)\n\n\ndef change_status(conn, ticket_id, expected, new, actor_id, now):\n    """Move a ticket from `expected` to `new`, unless someone changed its status first."""\n    with conn:\n        cur = conn.execute("UPDATE tickets SET status = ?, updated_at = ? WHERE id = ? AND status = ?",\n                           (new, now, ticket_id, expected))\n        if cur.rowcount != 1:\n            raise ConflictError(f"ticket {ticket_id} is no longer {expected}")\n        _event(conn, ticket_id, actor_id, "status", f"{expected}->{new}", now)\n    return get_ticket(conn, ticket_id)\n\n\ndef assign(conn, ticket_id, agent_id, actor_id, now):\n    agent = get_user(conn, agent_id)\n    if agent is None or agent["role"] not in ("agent", "admin"):\n        raise ValueError(f"user {agent_id} is not an agent")\n    with conn:\n        conn.execute("UPDATE tickets SET assignee_id = ?, updated_at = ? WHERE id = ?", (agent_id, now, ticket_id))\n        _event(conn, ticket_id, actor_id, "assigned", str(agent_id), now)\n    return get_ticket(conn, ticket_id)\n\n\ndef add_comment(conn, ticket_id, author_id, body, internal, now):\n    with conn:\n        cur = conn.execute("INSERT INTO comments (ticket_id, author_id, body, internal, created_at) VALUES (?, ?, ?, ?, ?)",\n                           (ticket_id, author_id, body, int(internal), now))\n        conn.execute("UPDATE tickets SET updated_at = ? WHERE id = ?", (now, ticket_id))\n    return _dict(conn.execute("SELECT * FROM comments WHERE id = ?", (cur.lastrowid,)).fetchone())\n\n\ndef list_comments(conn, ticket_id, include_internal):\n    sql = "SELECT * FROM comments WHERE ticket_id = ?" + ("" if include_internal else " AND internal = 0") + " ORDER BY id"\n    return [dict(r) for r in conn.execute(sql, (ticket_id,)).fetchall()]\n\n\ndef delete_ticket(conn, ticket_id):\n    with conn:\n        return conn.execute("DELETE FROM tickets WHERE id = ?", (ticket_id,)).rowcount == 1\n\n\ndef events(conn, ticket_id):\n    return [dict(r) for r in conn.execute("SELECT kind, detail, actor_id, at FROM events WHERE ticket_id = ? ORDER BY id", (ticket_id,))]\n\n\ndef stats(conn):\n    by_status = {s: 0 for s in ("open", "in_progress", "waiting", "resolved", "closed")}\n    by_status.update(dict(conn.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall()))\n    by_priority = {p: 0 for p in ("low", "normal", "high", "urgent")}\n    by_priority.update(dict(conn.execute("SELECT priority, COUNT(*) FROM tickets WHERE status != \'closed\' GROUP BY priority").fetchall()))\n    unassigned = conn.execute("SELECT COUNT(*) FROM tickets WHERE assignee_id IS NULL AND status != \'closed\'").fetchone()[0]\n    return {"by_status": by_status, "open_by_priority": by_priority, "unassigned": unassigned}\n')
with open('helpdesk_security.py', 'w', encoding='utf-8') as f:
    f.write('"""Passwords and tokens for the helpdesk (Week 79\'s tools, gathered in one place)."""\nimport base64\nimport hashlib\nimport hmac\nimport secrets\n\n\ndef pbkdf2_sha256(password, salt, iterations, dklen=32):\n    base = hmac.new(password, digestmod=hashlib.sha256)\n    out, block = b"", 1\n    while len(out) < dklen:\n        h = base.copy()\n        h.update(salt + block.to_bytes(4, "big"))\n        u = h.digest()\n        total = int.from_bytes(u, "big")\n        for _ in range(iterations - 1):\n            h = base.copy()\n            h.update(u)\n            u = h.digest()\n            total ^= int.from_bytes(u, "big")\n        out += total.to_bytes(32, "big")\n        block += 1\n    return out[:dklen]\n\n\ndef hash_password(password, iterations):\n    salt = secrets.token_bytes(16)\n    digest = pbkdf2_sha256(password.encode(), salt, iterations)\n    return f"pbkdf2_sha256${iterations}${base64.b64encode(salt).decode()}${base64.b64encode(digest).decode()}"\n\n\ndef check_password(password, stored):\n    try:\n        algorithm, iterations, salt, digest = stored.split("$")\n        expected = pbkdf2_sha256(password.encode(), base64.b64decode(salt), int(iterations))\n        return algorithm == "pbkdf2_sha256" and hmac.compare_digest(expected, base64.b64decode(digest))\n    except ValueError:\n        return False\n\n\ndef issue_token(user_id, expires, key):\n    """A signed bearer token: USER_ID.EXPIRES.SIGNATURE."""\n    payload = f"{user_id}.{expires}"\n    return f"{payload}.{hmac.new(key, payload.encode(), hashlib.sha256).hexdigest()}"\n\n\ndef verify_token(token, key, now):\n    """The user id in a valid, unexpired token, else None."""\n    parts = token.split(".")\n    if len(parts) != 3 or not parts[0].isdigit() or not parts[1].isdigit():\n        return None\n    payload = f"{parts[0]}.{parts[1]}"\n    expected = hmac.new(key, payload.encode(), hashlib.sha256).hexdigest()\n    if not hmac.compare_digest(parts[2], expected) or now >= int(parts[1]):\n        return None\n    return int(parts[0])\n')

**Predict before you run:** what will this print?



In [ ]:
import hashlib, hmac

key = b"k" * 32

def issue(user_id, expires):
    payload = f"{user_id}.{expires}"
    return payload + "." + hmac.new(key, payload.encode(), hashlib.sha256).hexdigest()[:8]

token = issue(7, 1800000000)
print(token)
user, expires, sig = token.split(".")
forged = f"1.{expires}.{sig}"
print(hmac.compare_digest(sig, issue(1, int(expires)).split(".")[2]), forged.split(".")[0])

### Exercise: Step 11: errors and health

In `create_app`, register an error handler for each code in `ERROR_CODES` (a loop with `@app.errorhandler(code)`; pass the code into the handler as a default argument, `def handle(e, code=code)`, so each handler keeps its own). For `500` with an exception that isn't an `HTTPException`, make an incident id with `secrets.token_hex(4)`, call `log(f"incident ID: TYPE: MESSAGE")`, and return `problem(500, "something went wrong", incident=ID)`; otherwise return `problem(e.code, e.description, **extra)`, where `extra` is the exception's `extra` dict if it has one (`getattr(e, "extra", {})`). Then add `GET /healthz` returning `{"status": "ok"}`, and `GET /readyz`: read `MAX(version)` from `schema_migrations` (any exception counts as not ready); if it equals `len(db.MIGRATIONS)`, return `{"status": "ready", "schema": VERSION}`, else raise `ApiError(503, "the database isn't ready")`.

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    # Step 11: your code here
    ___

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _app(migrated=True):\n    import helpdesk_db as db\n    conn = db.connect()\n    if migrated:\n        db.migrate(conn)\n    logs = []\n    app = create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50}, conn, lambda: 1800000000, logs.append)\n    return app, conn, logs\n\ndef test_errors():\n    \"every error is a problem\"\n    app, conn, logs = _app()\n    r = app.test_client().get(\"/nowhere\")\n    assert r.status_code == 404 and r.headers[\"Content-Type\"] == \"application/problem+json\"\n    assert r.get_json() == {\"type\": \"about:blank\", \"title\": \"Not Found\", \"status\": 404, \"detail\": \"The requested URL was not found on the server.\"}\n    r = app.test_client().post(\"/healthz\")\n    assert r.status_code == 405 and r.get_json()[\"status\"] == 405\n\n    @app.get(\"/boom\")\n    def boom():\n        raise RuntimeError(\"database password is hunter2\")\n\n    @app.get(\"/teapot\")\n    def teapot():\n        raise ApiError(409, \"no\", hint=\"reload\")\n\n    r = app.test_client().get(\"/boom\")\n    body = r.get_json()\n    assert r.status_code == 500 and set(body) == {\"type\", \"title\", \"status\", \"detail\", \"incident\"} and \"hunter2\" not in str(body)\n    assert len(body[\"incident\"]) == 8 and logs == [f\"incident {body['incident']}: RuntimeError: database password is hunter2\"]\n    assert app.test_client().get(\"/teapot\").get_json() == {\"type\": \"about:blank\", \"title\": \"Conflict\", \"status\": 409, \"detail\": \"no\", \"hint\": \"reload\"}\n\ndef test_health():\n    \"alive and ready\"\n    app, conn, logs = _app()\n    c = app.test_client()\n    assert c.get(\"/healthz\").get_json() == {\"status\": \"ok\"} and c.get(\"/readyz\").get_json() == {\"status\": \"ready\", \"schema\": 4}\n    app, conn, logs = _app(migrated=False)\n    r = app.test_client().get(\"/readyz\")\n    assert r.status_code == 503 and r.get_json()[\"detail\"] == \"the database isn't ready\"\n    import helpdesk_db as db\n    db.migrate(conn, db.MIGRATIONS[:2])\n    assert app.test_client().get(\"/readyz\").status_code == 503")

### Exercise: Step 12: accounts and tokens

Add the `before_request` hook (`g.user = None`, then, for a header starting `Bearer `, `verify_token(TOKEN, key, int(clock()))` and `db.get_user` if it gives an id); `user()` (the user, or `ApiError(401, "log in first")`); `body()` (`request.get_json(silent=True)`, or `ApiError(400, "send a JSON object")` unless it's a dict); `POST /users` (email from the body, if it's a string, stripped and lower-cased, matching `[^@\s]+@[^@\s]+\.[a-z]{2,}` in full, and a string password of 12 to 128 characters, else `ApiError(422, "a valid email and a password of 12 to 128 characters are needed")`; create the user with role `"customer"`, `hash_password(password, iterations)` and `iso(clock())`; a `ValueError` becomes `ApiError(409, "email already registered")`; return `{"id", "email", "role"}` with `201`); `POST /tokens` (look the user up by the stripped, lower-cased email; check the password against their hash, or `dummy` if there's no such user; failing either way raises `ApiError(401, "wrong email or password")`; otherwise return `{"token": issue_token(ID, EXPIRES, key), "expires_at": iso(EXPIRES)}` with `201`, where EXPIRES is `int(clock()) + token_seconds`); and `GET /me`.

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    for code in ERROR_CODES:
        @app.errorhandler(code)
        def handle(e, code=code):
            if code == 500 and not isinstance(e, HTTPException):
                incident = secrets.token_hex(4)
                log(f"incident {incident}: {type(e).__name__}: {e}")
                return problem(500, "something went wrong", incident=incident)
            return problem(e.code, e.description, **getattr(e, "extra", {}))

    @app.get("/healthz")
    def healthz():
        return {"status": "ok"}

    @app.get("/readyz")
    def readyz():
        try:
            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]
        except Exception:
            version = None
        if version != len(db.MIGRATIONS):
            raise ApiError(503, "the database isn't ready")
        return {"status": "ready", "schema": version}

    # Step 12: your code here
    ___

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _app(clock):\n    import helpdesk_db as db\n    conn = db.connect()\n    db.migrate(conn)\n    return create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50, \"TOKEN_SECONDS\": 600}, conn, clock, [].append), conn\n\ndef test_signup():\n    \"customers sign up\"\n    t = [1800000000]\n    app, conn = _app(lambda: t[0])\n    c = app.test_client()\n    r = c.post(\"/users\", json={\"email\": \" Ada@Example.ORG \", \"password\": \"correct horse battery\"})\n    assert r.status_code == 201 and r.get_json() == {\"id\": 1, \"email\": \"ada@example.org\", \"role\": \"customer\"}\n    stored = conn.execute(\"SELECT password FROM users\").fetchone()[0]\n    assert stored.startswith(\"pbkdf2_sha256$50$\") and \"correct horse\" not in stored\n    assert c.post(\"/users\", json={\"email\": \"ada@example.org\", \"password\": \"another long one\"}).status_code == 409\n    for bad in ({\"email\": \"ada\", \"password\": \"correct horse battery\"}, {\"email\": \"b@example.org\", \"password\": \"short\"},\n                {\"email\": 5, \"password\": \"correct horse battery\"}, {\"email\": \"c@example.org\"}):\n        r = c.post(\"/users\", json=bad)\n        assert r.status_code == 422 and r.get_json()[\"detail\"] == \"a valid email and a password of 12 to 128 characters are needed\", bad\n    assert c.post(\"/users\", data=\"not json\").status_code == 400 and c.post(\"/users\", json=[1]).get_json()[\"detail\"] == \"send a JSON object\"\n\ndef test_tokens():\n    \"log in, use the token, watch it expire\"\n    t = [1800000000]\n    app, conn = _app(lambda: t[0])\n    c = app.test_client()\n    c.post(\"/users\", json={\"email\": \"ada@example.org\", \"password\": \"correct horse battery\"})\n    assert c.get(\"/me\").status_code == 401 and c.get(\"/me\").get_json()[\"detail\"] == \"log in first\"\n    wrong = [c.post(\"/tokens\", json=b) for b in ({\"email\": \"ada@example.org\", \"password\": \"nope\"}, {\"email\": \"eve@example.org\", \"password\": \"nope\"})]\n    assert [r.status_code for r in wrong] == [401, 401] and wrong[0].get_json() == wrong[1].get_json()\n    r = c.post(\"/tokens\", json={\"email\": \"ADA@example.org\", \"password\": \"correct horse battery\"})\n    assert r.status_code == 201 and r.get_json()[\"expires_at\"] == \"2027-01-15T08:10:00Z\"\n    auth = {\"Authorization\": \"Bearer \" + r.get_json()[\"token\"]}\n    assert c.get(\"/me\", headers=auth).get_json() == {\"id\": 1, \"email\": \"ada@example.org\", \"role\": \"customer\", \"created_at\": \"2027-01-15T08:00:00Z\"}\n    user_id, expires, sig = r.get_json()[\"token\"].split(\".\")\n    for forged in (f\"2.{expires}.{sig}\", f\"1.{int(expires) + 999}.{sig}\", \"1.2.3\", \"garbage\"):\n        assert c.get(\"/me\", headers={\"Authorization\": \"Bearer \" + forged}).status_code == 401, forged\n    t[0] += 600\n    assert c.get(\"/me\", headers=auth).status_code == 401, \"tokens expire\"")

### Exercise: Step 13: opening and reading tickets

Add `checked`, `visible_ticket` and `require` as described (the messages: `"the request has invalid fields"` with `errors=e.errors` for 422; `f"no ticket {ticket_id}"` for 404). Then `POST /tickets` (requires `"create_ticket"`, message `"only customers can open tickets"`; creates the ticket from `checked(body())` with `iso(clock())`; returns it with `201` and a `Location: /tickets/ID` header); `GET /tickets/<int:ticket_id>` (the visible ticket); and `GET /tickets`: `status` and `priority`, if given, must be in `STATUSES` and the priorities (`400`, `"NAME must be one of A, B, ..."`); `after` (default `0`, from 0 to 1000000000) and `limit` (default `20`, from 1 to 50) must be digits within range (`400`, `"NAME must be a whole number from LOW to HIGH"`); users who may not `"list_all_tickets"` are limited to their own; `q` (if non-empty) searches titles. Return `{"items": ITEMS, "next": NEXT}` from `db.list_tickets`.

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    for code in ERROR_CODES:
        @app.errorhandler(code)
        def handle(e, code=code):
            if code == 500 and not isinstance(e, HTTPException):
                incident = secrets.token_hex(4)
                log(f"incident {incident}: {type(e).__name__}: {e}")
                return problem(500, "something went wrong", incident=incident)
            return problem(e.code, e.description, **getattr(e, "extra", {}))

    @app.get("/healthz")
    def healthz():
        return {"status": "ok"}

    @app.get("/readyz")
    def readyz():
        try:
            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]
        except Exception:
            version = None
        if version != len(db.MIGRATIONS):
            raise ApiError(503, "the database isn't ready")
        return {"status": "ready", "schema": version}

    @app.before_request
    def authenticate():
        g.user = None
        header = request.headers.get("Authorization", "")
        if header.startswith("Bearer "):
            user_id = verify_token(header[7:], key, int(clock()))
            if user_id is not None:
                g.user = db.get_user(conn, user_id)

    def user():
        if g.user is None:
            raise ApiError(401, "log in first")
        return g.user

    def body():
        data = request.get_json(silent=True)
        if not isinstance(data, dict):
            raise ApiError(400, "send a JSON object")
        return data

    @app.post("/users")
    def signup():
        data = body()
        email = data.get("email", "").strip().lower() if isinstance(data.get("email"), str) else ""
        password = data.get("password")
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", email) or not isinstance(password, str) or not 12 <= len(password) <= 128:
            raise ApiError(422, "a valid email and a password of 12 to 128 characters are needed")
        try:
            user_id = db.create_user(conn, email, hash_password(password, iterations), "customer", iso(clock()))
        except ValueError:
            raise ApiError(409, "email already registered") from None
        return {"id": user_id, "email": email, "role": "customer"}, 201

    @app.post("/tokens")
    def login():
        data = body()
        found = db.get_user_by_email(conn, str(data.get("email", "")).strip().lower())
        stored = found["password"] if found else dummy
        if not check_password(str(data.get("password", "")), stored) or found is None:
            raise ApiError(401, "wrong email or password")
        expires = int(clock()) + token_seconds
        return {"token": issue_token(found["id"], expires, key), "expires_at": iso(expires)}, 201

    @app.get("/me")
    def me():
        return user()

    # Step 13: your code here
    ___

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _setup():\n    import helpdesk_db as db\n    from helpdesk_security import hash_password\n    conn = db.connect()\n    db.migrate(conn)\n    app = create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50}, conn, lambda: 1800000000, [].append)\n    db.create_user(conn, \"grace@example.org\", hash_password(\"agent password 1\", 50), \"agent\", \"t\")\n    c = app.test_client()\n    tokens = {}\n    for email in (\"ada@example.org\", \"bob@example.org\"):\n        c.post(\"/users\", json={\"email\": email, \"password\": \"customer password\"})\n    for email, pw in ((\"ada@example.org\", \"customer password\"), (\"bob@example.org\", \"customer password\"), (\"grace@example.org\", \"agent password 1\")):\n        tokens[email.split(\"@\")[0]] = {\"Authorization\": \"Bearer \" + c.post(\"/tokens\", json={\"email\": email, \"password\": pw}).get_json()[\"token\"]}\n    return c, tokens\n\ndef test_open_and_read():\n    \"customers open; only owners and staff can read\"\n    c, h = _setup()\n    r = c.post(\"/tickets\", json={\"title\": \"Printer on fire\", \"body\": \"Smoke everywhere\", \"priority\": \"urgent\"}, headers=h[\"ada\"])\n    assert r.status_code == 201 and r.headers[\"Location\"] == \"/tickets/1\" and r.get_json()[\"status\"] == \"open\"\n    r = c.post(\"/tickets\", json={\"title\": \"x\", \"status\": \"closed\"}, headers=h[\"ada\"])\n    assert r.status_code == 422 and r.get_json()[\"errors\"] == {\"_\": \"unexpected field(s): status\", \"title\": \"3 to 120 characters\", \"body\": \"required\"}\n    r = c.post(\"/tickets\", json={\"title\": \"Agent ticket\", \"body\": \"x\"}, headers=h[\"grace\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only customers can open tickets\"\n    assert c.post(\"/tickets\", json={\"title\": \"abc\", \"body\": \"x\"}).status_code == 401\n    assert c.get(\"/tickets/1\", headers=h[\"ada\"]).get_json()[\"title\"] == \"Printer on fire\"\n    assert c.get(\"/tickets/1\", headers=h[\"grace\"]).status_code == 200\n    r = c.get(\"/tickets/1\", headers=h[\"bob\"])\n    assert r.status_code == 404 and r.get_json()[\"detail\"] == \"no ticket 1\" and c.get(\"/tickets/99\", headers=h[\"grace\"]).get_json()[\"detail\"] == \"no ticket 99\"\n\ndef test_list():\n    \"own tickets only, filters and pages\"\n    c, h = _setup()\n    for who, title, prio in ((\"ada\", \"Printer on fire\", \"urgent\"), (\"bob\", \"Mouse gone\", \"low\"), (\"ada\", \"Printer jam\", \"low\"), (\"ada\", \"Wifi down\", \"high\")):\n        c.post(\"/tickets\", json={\"title\": title, \"body\": \"x\", \"priority\": prio}, headers=h[who])\n    ids = lambda q, who: [t[\"id\"] for t in c.get(\"/tickets\" + q, headers=h[who]).get_json()[\"items\"]]\n    assert ids(\"\", \"ada\") == [1, 3, 4] and ids(\"\", \"bob\") == [2] and ids(\"\", \"grace\") == [1, 2, 3, 4]\n    assert ids(\"?priority=low\", \"grace\") == [2, 3] and ids(\"?q=printer\", \"ada\") == [1, 3] and ids(\"?q=mouse\", \"ada\") == []\n    assert ids(\"?status=open&priority=high\", \"ada\") == [4]\n    page = c.get(\"/tickets?limit=2\", headers=h[\"grace\"]).get_json()\n    assert [t[\"id\"] for t in page[\"items\"]] == [1, 2] and page[\"next\"] == 2\n    assert c.get(\"/tickets?limit=2&after=2\", headers=h[\"grace\"]).get_json()[\"next\"] is None\n    for q, detail in ((\"?status=lost\", \"status must be one of open, in_progress, waiting, resolved, closed\"),\n                      (\"?priority=asap\", \"priority must be one of low, normal, high, urgent\"),\n                      (\"?limit=0\", \"limit must be a whole number from 1 to 50\"), (\"?limit=51\", \"limit must be a whole number from 1 to 50\"),\n                      (\"?after=-1\", \"after must be a whole number from 0 to 1000000000\")):\n        r = c.get(\"/tickets\" + q, headers=h[\"ada\"])\n        assert r.status_code == 400 and r.get_json()[\"detail\"] == detail, q")

### Exercise: Step 14: changing tickets

Add `PATCH /tickets/<int:ticket_id>` (the visible ticket; `checked(body(), partial=True)`; a `priority` change requires `"change_priority"` (`"only staff can change the priority"`); a `title` or `body` change requires `"edit_ticket"` (`"only the customer who opened it can edit it"`), and for a customer the ticket must be `open`, else `ApiError(409, "only open tickets can be edited")`; return `db.update_ticket(...)`); `POST /tickets/<int:ticket_id>/transitions` (the new status from the body must be in `STATUSES`, else `422` with `"status must be one of open, in_progress, waiting, resolved, closed"`; if `can_transition` refuses, `403` with `"you can't move a ticket from OLD to NEW"`; `db.ConflictError` becomes `409` with `"the ticket changed; reload it and try again"`; return the ticket); and `PUT /tickets/<int:ticket_id>/assignee` (requires `"assign"`, `"only staff can assign tickets"`; `agent_id` must be an `int` and not a `bool`, else `422` `"agent_id must be a user id"`; a `ValueError` from `db.assign` becomes `422` with its message; return the ticket).

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    for code in ERROR_CODES:
        @app.errorhandler(code)
        def handle(e, code=code):
            if code == 500 and not isinstance(e, HTTPException):
                incident = secrets.token_hex(4)
                log(f"incident {incident}: {type(e).__name__}: {e}")
                return problem(500, "something went wrong", incident=incident)
            return problem(e.code, e.description, **getattr(e, "extra", {}))

    @app.get("/healthz")
    def healthz():
        return {"status": "ok"}

    @app.get("/readyz")
    def readyz():
        try:
            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]
        except Exception:
            version = None
        if version != len(db.MIGRATIONS):
            raise ApiError(503, "the database isn't ready")
        return {"status": "ready", "schema": version}

    @app.before_request
    def authenticate():
        g.user = None
        header = request.headers.get("Authorization", "")
        if header.startswith("Bearer "):
            user_id = verify_token(header[7:], key, int(clock()))
            if user_id is not None:
                g.user = db.get_user(conn, user_id)

    def user():
        if g.user is None:
            raise ApiError(401, "log in first")
        return g.user

    def body():
        data = request.get_json(silent=True)
        if not isinstance(data, dict):
            raise ApiError(400, "send a JSON object")
        return data

    @app.post("/users")
    def signup():
        data = body()
        email = data.get("email", "").strip().lower() if isinstance(data.get("email"), str) else ""
        password = data.get("password")
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", email) or not isinstance(password, str) or not 12 <= len(password) <= 128:
            raise ApiError(422, "a valid email and a password of 12 to 128 characters are needed")
        try:
            user_id = db.create_user(conn, email, hash_password(password, iterations), "customer", iso(clock()))
        except ValueError:
            raise ApiError(409, "email already registered") from None
        return {"id": user_id, "email": email, "role": "customer"}, 201

    @app.post("/tokens")
    def login():
        data = body()
        found = db.get_user_by_email(conn, str(data.get("email", "")).strip().lower())
        stored = found["password"] if found else dummy
        if not check_password(str(data.get("password", "")), stored) or found is None:
            raise ApiError(401, "wrong email or password")
        expires = int(clock()) + token_seconds
        return {"token": issue_token(found["id"], expires, key), "expires_at": iso(expires)}, 201

    @app.get("/me")
    def me():
        return user()

    def checked(data, partial=False):
        try:
            return validate_ticket(data, partial)
        except ValidationError as e:
            raise ApiError(422, "the request has invalid fields", errors=e.errors) from None

    def visible_ticket(ticket_id):
        me = user()
        ticket = db.get_ticket(conn, ticket_id)
        if ticket is None or not allowed(me["role"], "view_ticket", ticket["customer_id"] == me["id"]):
            raise ApiError(404, f"no ticket {ticket_id}")
        return me, ticket

    def require(me, action, ticket=None, message="you can't do that"):
        if not allowed(me["role"], action, ticket is not None and ticket["customer_id"] == me["id"]):
            raise ApiError(403, message)

    @app.post("/tickets")
    def open_ticket():
        me = user()
        require(me, "create_ticket", message="only customers can open tickets")
        ticket = db.create_ticket(conn, me["id"], checked(body()), iso(clock()))
        return ticket, 201, {"Location": f"/tickets/{ticket['id']}"}

    @app.get("/tickets")
    def list_tickets():
        me = user()
        args = request.args
        filters = {}
        for name, choices in (("status", STATUSES), ("priority", ("low", "normal", "high", "urgent"))):
            if name in args:
                if args[name] not in choices:
                    raise ApiError(400, f"{name} must be one of {', '.join(choices)}")
                filters[name] = args[name]
        for name, low, high, default in (("after", 0, 10 ** 9, 0), ("limit", 1, 50, 20)):
            value = args.get(name, str(default))
            if not value.isdigit() or not low <= int(value) <= high:
                raise ApiError(400, f"{name} must be a whole number from {low} to {high}")
            filters[name] = int(value)
        if not allowed(me["role"], "list_all_tickets"):
            filters["customer_id"] = me["id"]
        items, after = db.list_tickets(conn, customer_id=filters.get("customer_id"), status=filters.get("status"),
                                       priority=filters.get("priority"), q=args.get("q") or None,
                                       after_id=filters["after"], limit=filters["limit"])
        return {"items": items, "next": after}

    @app.get("/tickets/<int:ticket_id>")
    def read_ticket(ticket_id):
        return visible_ticket(ticket_id)[1]

    # Step 14: your code here
    ___

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _setup():\n    import helpdesk_db as db\n    from helpdesk_security import hash_password\n    conn = db.connect()\n    db.migrate(conn)\n    app = create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50}, conn, lambda: 1800000000, [].append)\n    db.create_user(conn, \"grace@example.org\", hash_password(\"agent password 1\", 50), \"agent\", \"t\")\n    c = app.test_client()\n    c.post(\"/users\", json={\"email\": \"ada@example.org\", \"password\": \"customer password\"})\n    h = {}\n    for name, pw in ((\"ada\", \"customer password\"), (\"grace\", \"agent password 1\")):\n        h[name] = {\"Authorization\": \"Bearer \" + c.post(\"/tokens\", json={\"email\": f\"{name}@example.org\", \"password\": pw}).get_json()[\"token\"]}\n    c.post(\"/tickets\", json={\"title\": \"Printer on fire\", \"body\": \"Smoke\"}, headers=h[\"ada\"])\n    return c, h, conn\n\ndef test_edit():\n    \"who may change what\"\n    c, h, conn = _setup()\n    assert c.patch(\"/tickets/1\", json={\"title\": \"Printer smoking\"}, headers=h[\"ada\"]).get_json()[\"title\"] == \"Printer smoking\"\n    r = c.patch(\"/tickets/1\", json={\"priority\": \"urgent\"}, headers=h[\"ada\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only staff can change the priority\"\n    assert c.patch(\"/tickets/1\", json={\"priority\": \"urgent\"}, headers=h[\"grace\"]).get_json()[\"priority\"] == \"urgent\"\n    r = c.patch(\"/tickets/1\", json={\"body\": \"agent rewrite\"}, headers=h[\"grace\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only the customer who opened it can edit it\"\n    assert c.patch(\"/tickets/1\", json={}, headers=h[\"ada\"]).status_code == 422\n    c.post(\"/tickets/1/transitions\", json={\"status\": \"in_progress\"}, headers=h[\"grace\"])\n    r = c.patch(\"/tickets/1\", json={\"body\": \"more detail\"}, headers=h[\"ada\"])\n    assert r.status_code == 409 and r.get_json()[\"detail\"] == \"only open tickets can be edited\"\n\ndef test_transitions_and_assignee():\n    \"the state machine over HTTP\"\n    c, h, conn = _setup()\n    r = c.post(\"/tickets/1/transitions\", json={\"status\": \"resolved\"}, headers=h[\"grace\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"you can't move a ticket from open to resolved\"\n    assert c.post(\"/tickets/1/transitions\", json={\"status\": \"lost\"}, headers=h[\"grace\"]).status_code == 422\n    assert c.post(\"/tickets/1/transitions\", json={\"status\": \"in_progress\"}, headers=h[\"ada\"]).status_code == 403\n    assert c.post(\"/tickets/1/transitions\", json={\"status\": \"in_progress\"}, headers=h[\"grace\"]).get_json()[\"status\"] == \"in_progress\"\n    assert c.post(\"/tickets/1/transitions\", json={\"status\": \"resolved\"}, headers=h[\"grace\"]).status_code == 200\n    assert c.post(\"/tickets/1/transitions\", json={\"status\": \"closed\"}, headers=h[\"ada\"]).get_json()[\"status\"] == \"closed\"\n    assert c.put(\"/tickets/1/assignee\", json={\"agent_id\": 2}, headers=h[\"grace\"]).get_json()[\"detail\"] == \"user 2 is not an agent\"\n    r = c.put(\"/tickets/1/assignee\", json={\"agent_id\": 1}, headers=h[\"ada\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only staff can assign tickets\"\n    assert c.put(\"/tickets/1/assignee\", json={\"agent_id\": True}, headers=h[\"grace\"]).get_json()[\"detail\"] == \"agent_id must be a user id\"\n    assert c.put(\"/tickets/1/assignee\", json={\"agent_id\": 1}, headers=h[\"grace\"]).get_json()[\"assignee_id\"] == 1\n\ndef test_conflict():\n    \"a status that changed meanwhile\"\n    import helpdesk_db as db\n    c, h, conn = _setup()\n    real = db.change_status\n    def racing(conn_, ticket_id, expected, new, actor_id, now):\n        real(conn_, ticket_id, \"open\", \"closed\", 1, now)\n        return real(conn_, ticket_id, expected, new, actor_id, now)\n    db.change_status = racing\n    try:\n        r = c.post(\"/tickets/1/transitions\", json={\"status\": \"in_progress\"}, headers=h[\"grace\"])\n    finally:\n        db.change_status = real\n    assert r.status_code == 409 and r.get_json()[\"detail\"] == \"the ticket changed; reload it and try again\"")

### Exercise: Step 15: comments, statistics and deletion

Add `DELETE /tickets/<int:ticket_id>` (visible ticket; requires `"delete_ticket"`, `"only admins can delete tickets"`; delete it and return `("", 204)`); `GET /tickets/<int:ticket_id>/comments` (`{"items": ...}` from `db.list_comments`, including internal notes only if the user is `allowed` the `"internal_note"` action); `POST /tickets/<int:ticket_id>/comments` (requires `"comment"`; the body's `body` must be a string of 1 to 5000 characters after stripping and `internal` (default `False`) a `bool`, else `422` `"body must be 1 to 5000 characters and internal true or false"`; an internal note requires `"internal_note"`, `"only staff can add internal notes"`; return the comment with `201`); and `GET /stats` (requires `"view_stats"`, `"only staff can see statistics"`; return `db.stats(conn)`).

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    for code in ERROR_CODES:
        @app.errorhandler(code)
        def handle(e, code=code):
            if code == 500 and not isinstance(e, HTTPException):
                incident = secrets.token_hex(4)
                log(f"incident {incident}: {type(e).__name__}: {e}")
                return problem(500, "something went wrong", incident=incident)
            return problem(e.code, e.description, **getattr(e, "extra", {}))

    @app.get("/healthz")
    def healthz():
        return {"status": "ok"}

    @app.get("/readyz")
    def readyz():
        try:
            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]
        except Exception:
            version = None
        if version != len(db.MIGRATIONS):
            raise ApiError(503, "the database isn't ready")
        return {"status": "ready", "schema": version}

    @app.before_request
    def authenticate():
        g.user = None
        header = request.headers.get("Authorization", "")
        if header.startswith("Bearer "):
            user_id = verify_token(header[7:], key, int(clock()))
            if user_id is not None:
                g.user = db.get_user(conn, user_id)

    def user():
        if g.user is None:
            raise ApiError(401, "log in first")
        return g.user

    def body():
        data = request.get_json(silent=True)
        if not isinstance(data, dict):
            raise ApiError(400, "send a JSON object")
        return data

    @app.post("/users")
    def signup():
        data = body()
        email = data.get("email", "").strip().lower() if isinstance(data.get("email"), str) else ""
        password = data.get("password")
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", email) or not isinstance(password, str) or not 12 <= len(password) <= 128:
            raise ApiError(422, "a valid email and a password of 12 to 128 characters are needed")
        try:
            user_id = db.create_user(conn, email, hash_password(password, iterations), "customer", iso(clock()))
        except ValueError:
            raise ApiError(409, "email already registered") from None
        return {"id": user_id, "email": email, "role": "customer"}, 201

    @app.post("/tokens")
    def login():
        data = body()
        found = db.get_user_by_email(conn, str(data.get("email", "")).strip().lower())
        stored = found["password"] if found else dummy
        if not check_password(str(data.get("password", "")), stored) or found is None:
            raise ApiError(401, "wrong email or password")
        expires = int(clock()) + token_seconds
        return {"token": issue_token(found["id"], expires, key), "expires_at": iso(expires)}, 201

    @app.get("/me")
    def me():
        return user()

    def checked(data, partial=False):
        try:
            return validate_ticket(data, partial)
        except ValidationError as e:
            raise ApiError(422, "the request has invalid fields", errors=e.errors) from None

    def visible_ticket(ticket_id):
        me = user()
        ticket = db.get_ticket(conn, ticket_id)
        if ticket is None or not allowed(me["role"], "view_ticket", ticket["customer_id"] == me["id"]):
            raise ApiError(404, f"no ticket {ticket_id}")
        return me, ticket

    def require(me, action, ticket=None, message="you can't do that"):
        if not allowed(me["role"], action, ticket is not None and ticket["customer_id"] == me["id"]):
            raise ApiError(403, message)

    @app.post("/tickets")
    def open_ticket():
        me = user()
        require(me, "create_ticket", message="only customers can open tickets")
        ticket = db.create_ticket(conn, me["id"], checked(body()), iso(clock()))
        return ticket, 201, {"Location": f"/tickets/{ticket['id']}"}

    @app.get("/tickets")
    def list_tickets():
        me = user()
        args = request.args
        filters = {}
        for name, choices in (("status", STATUSES), ("priority", ("low", "normal", "high", "urgent"))):
            if name in args:
                if args[name] not in choices:
                    raise ApiError(400, f"{name} must be one of {', '.join(choices)}")
                filters[name] = args[name]
        for name, low, high, default in (("after", 0, 10 ** 9, 0), ("limit", 1, 50, 20)):
            value = args.get(name, str(default))
            if not value.isdigit() or not low <= int(value) <= high:
                raise ApiError(400, f"{name} must be a whole number from {low} to {high}")
            filters[name] = int(value)
        if not allowed(me["role"], "list_all_tickets"):
            filters["customer_id"] = me["id"]
        items, after = db.list_tickets(conn, customer_id=filters.get("customer_id"), status=filters.get("status"),
                                       priority=filters.get("priority"), q=args.get("q") or None,
                                       after_id=filters["after"], limit=filters["limit"])
        return {"items": items, "next": after}

    @app.get("/tickets/<int:ticket_id>")
    def read_ticket(ticket_id):
        return visible_ticket(ticket_id)[1]

    @app.patch("/tickets/<int:ticket_id>")
    def edit_ticket(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        changes = checked(body(), partial=True)
        if "priority" in changes:
            require(me, "change_priority", ticket, "only staff can change the priority")
        if "title" in changes or "body" in changes:
            require(me, "edit_ticket", ticket, "only the customer who opened it can edit it")
            if me["role"] == "customer" and ticket["status"] != "open":
                raise ApiError(409, "only open tickets can be edited")
        return db.update_ticket(conn, ticket_id, changes, me["id"], iso(clock()))

    @app.post("/tickets/<int:ticket_id>/transitions")
    def transition(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        new = body().get("status")
        if new not in STATUSES:
            raise ApiError(422, f"status must be one of {', '.join(STATUSES)}")
        if not can_transition(ticket["status"], new, me["role"]):
            raise ApiError(403, f"you can't move a ticket from {ticket['status']} to {new}")
        try:
            return db.change_status(conn, ticket_id, ticket["status"], new, me["id"], iso(clock()))
        except db.ConflictError:
            raise ApiError(409, "the ticket changed; reload it and try again") from None

    @app.put("/tickets/<int:ticket_id>/assignee")
    def assign(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        require(me, "assign", ticket, "only staff can assign tickets")
        agent_id = body().get("agent_id")
        if not isinstance(agent_id, int) or isinstance(agent_id, bool):
            raise ApiError(422, "agent_id must be a user id")
        try:
            return db.assign(conn, ticket_id, agent_id, me["id"], iso(clock()))
        except ValueError as e:
            raise ApiError(422, str(e)) from None

    # Step 15: your code here
    ___

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _setup():\n    import helpdesk_db as db\n    from helpdesk_security import hash_password\n    conn = db.connect()\n    db.migrate(conn)\n    app = create_app({\"SECRET_KEY\": \"k\" * 40, \"PASSWORD_ITERATIONS\": 50}, conn, lambda: 1800000000, [].append)\n    db.create_user(conn, \"grace@example.org\", hash_password(\"agent password 1\", 50), \"agent\", \"t\")\n    db.create_user(conn, \"root@example.org\", hash_password(\"admin password 1\", 50), \"admin\", \"t\")\n    c = app.test_client()\n    c.post(\"/users\", json={\"email\": \"ada@example.org\", \"password\": \"customer password\"})\n    h = {}\n    for name, pw in ((\"ada\", \"customer password\"), (\"grace\", \"agent password 1\"), (\"root\", \"admin password 1\")):\n        h[name] = {\"Authorization\": \"Bearer \" + c.post(\"/tokens\", json={\"email\": f\"{name}@example.org\", \"password\": pw}).get_json()[\"token\"]}\n    c.post(\"/tickets\", json={\"title\": \"Printer on fire\", \"body\": \"Smoke\"}, headers=h[\"ada\"])\n    return c, h, conn\n\ndef test_comments():\n    \"internal notes stay internal\"\n    c, h, conn = _setup()\n    r = c.post(\"/tickets/1/comments\", json={\"body\": \"  Unplug it, please  \"}, headers=h[\"grace\"])\n    assert r.status_code == 201 and r.get_json()[\"body\"] == \"Unplug it, please\" and r.get_json()[\"internal\"] == 0\n    assert c.post(\"/tickets/1/comments\", json={\"body\": \"Ancient printer\", \"internal\": True}, headers=h[\"grace\"]).status_code == 201\n    r = c.post(\"/tickets/1/comments\", json={\"body\": \"secret\", \"internal\": True}, headers=h[\"ada\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only staff can add internal notes\"\n    assert c.post(\"/tickets/1/comments\", json={\"body\": \"Done, thanks\"}, headers=h[\"ada\"]).status_code == 201\n    for bad in ({\"body\": \"   \"}, {\"body\": \"x\", \"internal\": \"yes\"}, {\"internal\": False}):\n        assert c.post(\"/tickets/1/comments\", json=bad, headers=h[\"ada\"]).get_json()[\"detail\"] == \"body must be 1 to 5000 characters and internal true or false\"\n    assert [x[\"body\"] for x in c.get(\"/tickets/1/comments\", headers=h[\"ada\"]).get_json()[\"items\"]] == [\"Unplug it, please\", \"Done, thanks\"]\n    assert len(c.get(\"/tickets/1/comments\", headers=h[\"grace\"]).get_json()[\"items\"]) == 3\n\ndef test_stats_and_delete():\n    \"staff statistics; admin deletion\"\n    c, h, conn = _setup()\n    assert c.get(\"/stats\", headers=h[\"grace\"]).get_json()[\"by_status\"][\"open\"] == 1\n    r = c.get(\"/stats\", headers=h[\"ada\"])\n    assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only staff can see statistics\"\n    for who in (\"ada\", \"grace\"):\n        r = c.delete(\"/tickets/1\", headers=h[who])\n        assert r.status_code == 403 and r.get_json()[\"detail\"] == \"only admins can delete tickets\"\n    c.post(\"/tickets/1/comments\", json={\"body\": \"x\"}, headers=h[\"grace\"])\n    r = c.delete(\"/tickets/1\", headers=h[\"root\"])\n    assert r.status_code == 204 and r.data == b\"\"\n    assert c.get(\"/tickets/1\", headers=h[\"root\"]).status_code == 404\n    assert conn.execute(\"SELECT count(*) FROM comments\").fetchone()[0] == 0 and conn.execute(\"SELECT count(*) FROM events\").fetchone()[0] == 0")

In [ ]:
"""The helpdesk's HTTP API (Week 80, Day 3), built with miniflask (Week 77)."""
import re
import secrets
import time
from datetime import datetime, timezone

import helpdesk_db as db
from helpdesk_domain import ROLES, STATUSES, ValidationError, allowed, can_transition, problem, validate_ticket
from helpdesk_security import check_password, hash_password, issue_token, verify_token
from miniflask import Flask, HTTPException, g, request

ERROR_CODES = (400, 401, 403, 404, 405, 409, 415, 422, 500, 503)


class ApiError(HTTPException):
    def __init__(self, code, detail, **extra):
        super().__init__(code, detail)
        self.extra = extra


def iso(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def create_app(settings, conn, clock=time.time, log=print):
    app = Flask(__name__)
    key = settings["SECRET_KEY"].encode()
    token_seconds = settings.get("TOKEN_SECONDS", 3600)
    iterations = settings.get("PASSWORD_ITERATIONS", 600_000)
    dummy = hash_password(secrets.token_hex(8), iterations)   # checked for unknown emails, so they take as long as real ones

    for code in ERROR_CODES:
        @app.errorhandler(code)
        def handle(e, code=code):
            if code == 500 and not isinstance(e, HTTPException):
                incident = secrets.token_hex(4)
                log(f"incident {incident}: {type(e).__name__}: {e}")
                return problem(500, "something went wrong", incident=incident)
            return problem(e.code, e.description, **getattr(e, "extra", {}))

    @app.get("/healthz")
    def healthz():
        return {"status": "ok"}

    @app.get("/readyz")
    def readyz():
        try:
            version = conn.execute("SELECT MAX(version) FROM schema_migrations").fetchone()[0]
        except Exception:
            version = None
        if version != len(db.MIGRATIONS):
            raise ApiError(503, "the database isn't ready")
        return {"status": "ready", "schema": version}

    @app.before_request
    def authenticate():
        g.user = None
        header = request.headers.get("Authorization", "")
        if header.startswith("Bearer "):
            user_id = verify_token(header[7:], key, int(clock()))
            if user_id is not None:
                g.user = db.get_user(conn, user_id)

    def user():
        if g.user is None:
            raise ApiError(401, "log in first")
        return g.user

    def body():
        data = request.get_json(silent=True)
        if not isinstance(data, dict):
            raise ApiError(400, "send a JSON object")
        return data

    @app.post("/users")
    def signup():
        data = body()
        email = data.get("email", "").strip().lower() if isinstance(data.get("email"), str) else ""
        password = data.get("password")
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", email) or not isinstance(password, str) or not 12 <= len(password) <= 128:
            raise ApiError(422, "a valid email and a password of 12 to 128 characters are needed")
        try:
            user_id = db.create_user(conn, email, hash_password(password, iterations), "customer", iso(clock()))
        except ValueError:
            raise ApiError(409, "email already registered") from None
        return {"id": user_id, "email": email, "role": "customer"}, 201

    @app.post("/tokens")
    def login():
        data = body()
        found = db.get_user_by_email(conn, str(data.get("email", "")).strip().lower())
        stored = found["password"] if found else dummy
        if not check_password(str(data.get("password", "")), stored) or found is None:
            raise ApiError(401, "wrong email or password")
        expires = int(clock()) + token_seconds
        return {"token": issue_token(found["id"], expires, key), "expires_at": iso(expires)}, 201

    @app.get("/me")
    def me():
        return user()

    def checked(data, partial=False):
        try:
            return validate_ticket(data, partial)
        except ValidationError as e:
            raise ApiError(422, "the request has invalid fields", errors=e.errors) from None

    def visible_ticket(ticket_id):
        me = user()
        ticket = db.get_ticket(conn, ticket_id)
        if ticket is None or not allowed(me["role"], "view_ticket", ticket["customer_id"] == me["id"]):
            raise ApiError(404, f"no ticket {ticket_id}")
        return me, ticket

    def require(me, action, ticket=None, message="you can't do that"):
        if not allowed(me["role"], action, ticket is not None and ticket["customer_id"] == me["id"]):
            raise ApiError(403, message)

    @app.post("/tickets")
    def open_ticket():
        me = user()
        require(me, "create_ticket", message="only customers can open tickets")
        ticket = db.create_ticket(conn, me["id"], checked(body()), iso(clock()))
        return ticket, 201, {"Location": f"/tickets/{ticket['id']}"}

    @app.get("/tickets")
    def list_tickets():
        me = user()
        args = request.args
        filters = {}
        for name, choices in (("status", STATUSES), ("priority", ("low", "normal", "high", "urgent"))):
            if name in args:
                if args[name] not in choices:
                    raise ApiError(400, f"{name} must be one of {', '.join(choices)}")
                filters[name] = args[name]
        for name, low, high, default in (("after", 0, 10 ** 9, 0), ("limit", 1, 50, 20)):
            value = args.get(name, str(default))
            if not value.isdigit() or not low <= int(value) <= high:
                raise ApiError(400, f"{name} must be a whole number from {low} to {high}")
            filters[name] = int(value)
        if not allowed(me["role"], "list_all_tickets"):
            filters["customer_id"] = me["id"]
        items, after = db.list_tickets(conn, customer_id=filters.get("customer_id"), status=filters.get("status"),
                                       priority=filters.get("priority"), q=args.get("q") or None,
                                       after_id=filters["after"], limit=filters["limit"])
        return {"items": items, "next": after}

    @app.get("/tickets/<int:ticket_id>")
    def read_ticket(ticket_id):
        return visible_ticket(ticket_id)[1]

    @app.patch("/tickets/<int:ticket_id>")
    def edit_ticket(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        changes = checked(body(), partial=True)
        if "priority" in changes:
            require(me, "change_priority", ticket, "only staff can change the priority")
        if "title" in changes or "body" in changes:
            require(me, "edit_ticket", ticket, "only the customer who opened it can edit it")
            if me["role"] == "customer" and ticket["status"] != "open":
                raise ApiError(409, "only open tickets can be edited")
        return db.update_ticket(conn, ticket_id, changes, me["id"], iso(clock()))

    @app.post("/tickets/<int:ticket_id>/transitions")
    def transition(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        new = body().get("status")
        if new not in STATUSES:
            raise ApiError(422, f"status must be one of {', '.join(STATUSES)}")
        if not can_transition(ticket["status"], new, me["role"]):
            raise ApiError(403, f"you can't move a ticket from {ticket['status']} to {new}")
        try:
            return db.change_status(conn, ticket_id, ticket["status"], new, me["id"], iso(clock()))
        except db.ConflictError:
            raise ApiError(409, "the ticket changed; reload it and try again") from None

    @app.put("/tickets/<int:ticket_id>/assignee")
    def assign(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        require(me, "assign", ticket, "only staff can assign tickets")
        agent_id = body().get("agent_id")
        if not isinstance(agent_id, int) or isinstance(agent_id, bool):
            raise ApiError(422, "agent_id must be a user id")
        try:
            return db.assign(conn, ticket_id, agent_id, me["id"], iso(clock()))
        except ValueError as e:
            raise ApiError(422, str(e)) from None

    @app.delete("/tickets/<int:ticket_id>")
    def remove_ticket(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        require(me, "delete_ticket", ticket, "only admins can delete tickets")
        db.delete_ticket(conn, ticket_id)
        return "", 204

    @app.get("/tickets/<int:ticket_id>/comments")
    def comments(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        return {"items": db.list_comments(conn, ticket_id, include_internal=allowed(me["role"], "internal_note"))}

    @app.post("/tickets/<int:ticket_id>/comments")
    def comment(ticket_id):
        me, ticket = visible_ticket(ticket_id)
        require(me, "comment", ticket)
        data = body()
        text, internal = data.get("body"), data.get("internal", False)
        if not isinstance(text, str) or not 1 <= len(text.strip()) <= 5000 or not isinstance(internal, bool):
            raise ApiError(422, "body must be 1 to 5000 characters and internal true or false")
        if internal:
            require(me, "internal_note", ticket, "only staff can add internal notes")
        return db.add_comment(conn, ticket_id, me["id"], text.strip(), internal, iso(clock())), 201

    @app.get("/stats")
    def stats():
        require(user(), "view_stats", message="only staff can see statistics")
        return db.stats(conn)

    return app


import helpdesk_db as db
from helpdesk_security import hash_password

conn = db.connect()
db.migrate(conn)
app = create_app({"SECRET_KEY": "k" * 40, "PASSWORD_ITERATIONS": 50}, conn, lambda: 1800000000, print)
db.create_user(conn, "grace@example.org", hash_password("agent password 1", 50), "agent", "2027-01-15T08:00:00Z")
c = app.test_client()

def login(email, password):
    token = c.post("/tokens", json={"email": email, "password": password}).get_json()["token"]
    return {"Authorization": "Bearer " + token}

print(c.post("/users", json={"email": "ada@example.org", "password": "correct horse battery"}).status_code)
ada, grace = login("ada@example.org", "correct horse battery"), login("grace@example.org", "agent password 1")
ticket = c.post("/tickets", json={"title": "Printer on fire", "body": "Smoke everywhere", "priority": "urgent"}, headers=ada).get_json()
for who, status in ((grace, "in_progress"), (grace, "resolved"), (ada, "closed")):
    r = c.post(f"/tickets/{ticket['id']}/transitions", json={"status": status}, headers=who)
    print(r.status_code, r.get_json()["status"])
print(c.post(f"/tickets/{ticket['id']}/transitions", json={"status": "open"}, headers=grace).get_json()["detail"])
print(c.get("/stats", headers=grace).get_json()["by_status"])

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The files panel holds everything so far: `helpdesk_domain` (the rules), `helpdesk_db` (the database), `helpdesk_security` (password hashing and tokens, Week 79's tools gathered in one module) and `miniflask`. Today's module is `helpdesk_app`, with one function, `create_app(settings, conn, clock, log)`. Everything it depends on is a parameter (Week 77's app factory): the settings, the database connection, the clock (so tests can move time forward to expire a token) and the log function (so tests can read what was logged).

The tokens are simpler than Week 77's JWTs but built the same way: `USER_ID.EXPIRES.SIGNATURE`, the signature an HMAC of the first two parts with the secret key.

Each step today gives you the app as it stands, with a marked place for the new code. The checks call `create_app` with a fresh in-memory database, a fake clock, a low password work factor (so they run quickly) and a list to collect the log.

## Step 11: errors and health

Before any feature, decide what every **failure** looks like, so no endpoint has to. Two pieces:

- An exception, **`ApiError(code, detail, **extra)`**, a subclass of miniflask's `HTTPException`, that any view can raise: `raise ApiError(404, "no ticket 7")`. Raising (instead of returning an error) means helper functions such as "load this ticket or fail" can stop a request from anywhere.
- An **error handler** for each status code the API uses, turning the exception into `problem(...)` from the domain module. The `500` handler is special: it also receives exceptions that **aren't** `HTTPException`s (a bug, a database failure), and must not show them to the client: log `incident ID: TYPE: MESSAGE` and return only the incident id (Week 79).

Then two endpoints for whoever runs the service (Week 77 Day 5): **`/healthz`** says the process is alive, and **`/readyz`** says it can do its job, here meaning the database answers and every migration has been applied; otherwise `503`, so a load balancer sends traffic elsewhere until it's ready.

## Step 12: accounts and tokens

Three endpoints and one hook give the helpdesk its accounts:

- **`POST /users`** signs up a **customer** (staff accounts are created by admins, never through a public form): a valid email, lower-cased, and a password of 12 to 128 characters, else `422`; a taken email is `409`. The password is stored with `hash_password` (Week 79), never as it was sent.
- **`POST /tokens`** logs in: the right password returns `201` with `{"token", "expires_at"}`, where the token expires after `TOKEN_SECONDS`; a wrong email **or** password gives the same `401`, and an unknown email is still checked against a dummy hash, so both take the same time.
- A **`before_request`** hook reads `Authorization: Bearer TOKEN` on every request and, if the token is valid and unexpired, loads the user into `g.user`; otherwise `g.user` is `None`. Endpoints that need a user call `user()`, which raises `401` if there isn't one. Expiry uses the injected clock, so a test can step past it.
- **`GET /me`** returns the logged-in user (from `db.get_user`, which never includes the password).

A small helper, `body()`, reads the JSON body and raises `400` if it isn't a JSON object, so no view crashes on `None.get`.

## Step 13: opening and reading tickets

Now the tickets themselves, with the permission rules from Monday. Three helpers keep every endpoint short and consistent:

- **`checked(data, partial)`** runs `validate_ticket` and turns a `ValidationError` into a `422` whose `errors` field lists the problems.
- **`visible_ticket(ticket_id)`** loads a ticket for the current user, and raises **`404`** if it doesn't exist **or** the user may not view it, so a customer can't even confirm that someone else's ticket exists.
- **`require(me, action, ticket, message)`** raises `403` with a message unless `allowed(...)` says yes (with `is_owner` worked out from the ticket).

The list endpoint validates its query parameters (the status and priority must be known values; `after` and `limit` whole numbers in range, `limit` from 1 to 50), and **customers only ever get their own tickets**, whatever they ask for: the web layer adds `customer_id` itself, rather than trusting a parameter.

## Step 14: changing tickets

Three endpoints change tickets, each mapping a rule onto a status code:

- **`PATCH /tickets/{id}`** edits the title, body or priority. Only staff may change the **priority**; only the customer who opened it (or an admin) may change the **title or body**, and a customer only while it's still `open` (`409` once someone is working on it, because the request conflicts with the ticket's current state).
- **`POST /tickets/{id}/transitions`** with `{"status": NEW}` moves a ticket, if Monday's state machine allows that role to make that move (`403` otherwise, saying which move was refused); an unknown status is `422`. It passes the status the ticket had when it was loaded as `expected`, so if someone else moved it in the meantime, `ConflictError` becomes `409`.
- **`PUT /tickets/{id}/assignee`** with `{"agent_id": N}` assigns it (staff only); an id that isn't a whole number, or isn't staff, is `422`.

## Step 15: comments, statistics and deletion

The last endpoints:

- **`GET /tickets/{id}/comments`** lists a ticket's comments, with **internal notes only for staff**: the filter happens in the query, so a customer's response never contains them at all.
- **`POST /tickets/{id}/comments`** with `{"body", "internal"}` adds one (the owner or staff may comment; only staff may add internal notes).
- **`GET /stats`** returns Tuesday's statistics, to staff only.
- **`DELETE /tickets/{id}`** (admins only) answers `204` with an empty body; the database cascades to the ticket's comments and events.

## Checkpoint

The whole API, from sign-up to a closed ticket, as a customer and an agent would use it:

## Quiz

## Before you go

Look at the finished `create_app` and count how many lines are about HTTP and how many are rules; the rules should be the calls into `helpdesk_domain`. Tomorrow: **tests and continuous integration**, making sure all of this keeps working as the code changes.